In [5]:
import tensorflow as tf
import time
import os

# Suppress TensorFlow INFO messages (like the one you saw)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

print(f"--- Using TensorFlow Version: {tf.__version__} ---")

# --- 1. Check for GPU and enable memory growth ---
# This is a crucial step to ensure TF doesn't allocate all GPU memory at once
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(f"✅ Found {len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs.")
        print("GPU is available and ready! 🚀\n")
    except RuntimeError as e:
        print(f"⚠️ Error setting up GPU: {e}")
else:
    print("⚠️ No GPU found. This script will run on the CPU (which will be much slower).\n")

# --- 2. Load and Prepare the MNIST Dataset ---
print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values from 0-255 to 0.0-1.0
# Add a 'channel' dimension (for grayscale)
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
print("Dataset loaded and preprocessed.")
print(f"Training data shape: {x_train.shape}")

# --- 3. Build a Simple CNN Model ---
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax') # 10 classes for digits 0-9
])

# --- 4. Compile the Model ---
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("\nModel built and compiled. Summary:")
model.summary()

# --- 5. Train the Model (This is the GPU-heavy part) ---
print("\n--- 🚀 Starting Model Training (GPU Stress Test) ---")
print("Watch your GPU monitor now!")

NUM_EPOCHS = 15  # Set this higher (e.g., 50) to run for longer
BATCH_SIZE = 128 # Larger batches tend to use the GPU more effectively

start_time = time.time()

model.fit(x_train, y_train,
          epochs=NUM_EPOCHS,
          batch_size=BATCH_SIZE,
          validation_data=(x_test, y_test),
          verbose=1) # verbose=1 shows the live progress bar

end_time = time.time()
print("--- ✅ Training Finished ---")
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

--- Using TensorFlow Version: 2.19.0 ---
✅ Found 1 Physical GPUs, 1 Logical GPUs.
GPU is available and ready! 🚀

Loading MNIST dataset...


I0000 00:00:1761316491.324626    1000 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3537 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Dataset loaded and preprocessed.
Training data shape: (60000, 28, 28, 1)


/home/user/miniconda3/envs/tf_py12/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Model built and compiled. Summary:


...

 37/469 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4864 - loss: 1.7120

I0000 00:00:1761316495.936294    1065 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


461/469 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8502 - loss: 0.5136

2025-10-24 22:34:58.635455: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_466', 88 bytes spill stores, 88 bytes spill loads



469/469 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9358 - loss: 0.2192 - val_accuracy: 0.9788 - val_loss: 0.0646
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9823 - loss: 0.0592 - val_accuracy: 0.9871 - val_loss: 0.0393
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9871 - loss: 0.0413 - val_accuracy: 0.9866 - val_loss: 0.0405
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9902 - loss: 0.0320 - val_accuracy: 0.9880 - val_loss: 0.0317
Epoch 5/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9920 - loss: 0.0250 - val_accuracy: 0.9901 - val_loss: 0.0295
Epoch 6/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9937 - loss: 0.0198 - val_accuracy: 0.9905 - val_loss: 0.0325
Epoch 7/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9948 - loss: 0.0159 - val_accuracy: 0.9916 - val_loss: 0.0275
Epoch 8/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9966 - loss: 0.0117 - val_accuracy: 0.9909 - va